## Install Dependencies

Installs the required Python packages for this notebook:
- **transformers** – Hugging Face library for loading and running pre-trained models.
- **torch** – PyTorch deep learning framework (CUDA 11.8 build for GPU support).
- **tqdm** – Progress bar utility used by transformers during downloads.
- **azure-ai-ml** – Azure Machine Learning SDK v2 for managing models, endpoints, and deployments.
- **azure-identity** / **azure-keyvault-secrets** – Azure SDK libraries for credential management and reading secrets from Key Vault.

In [ ]:
%pip install transformers
%pip install torch==2.6.0 --index-url https://download.pytorch.org/whl/cu118
%pip install tqdm
%pip install azure-ai-ml azure-identity azure-keyvault-secrets

## Verify GPU Availability

Runs `nvidia-smi` to confirm that a CUDA-capable GPU is visible in the current compute environment. Review the output to check the GPU model, driver version, and available VRAM before proceeding with model download and inference.

In [ ]:
!nvidia-smi

## Configure Azure ML Client and Load Secrets from Key Vault

Initialises the Azure ML client using `DefaultAzureCredential` and the workspace config file, then retrieves all deployment configuration values from Azure Key Vault. Each secret (model ID, model name, scoring script, instance type, name prefixes, subscription, resource group, and workspace) can be overridden at runtime via environment variables, allowing the notebook to be driven from a CI/CD pipeline without modifying source code. The resolved values are printed for verification.

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient
import  os

AZURE_ML_MODEL_EMPTY_ID_SECRET_NAME="AZURE-ML-MODEL-EMPTY-ID"
AZURE_ML_MODEL_EMPTY_NAME_SECRET_NAME="AZURE-ML-MODEL-EMPTY-NAME"
AZURE_ML_MODEL_EMPTY_SCORE_SCRIPT_SECRET_NAME="AZURE-ML-MODEL-EMPTY-SCORE-SCRIPT"
AZURE_ML_MODEL_EMPTY_INSTANCE_TYPE_SECRET_NAME="AZURE-ML-MODEL-EMPTY-INSTANCE-TYPE" 

AZURE_ML_MODEL_PREFIX_SECRET_NAME="AZURE-ML-MODEL-PREFIX" 
AZURE_ML_ENDPOINT_PREFIX_SECRET_NAME="AZURE-ML-ENDPOINT-PREFIX" 
AZURE_ML_ENV_PREFIX_SECRET_NAME="AZURE-ML-ENV-PREFIX" 
AZURE_ML_SUBSCRIPTION_ID_SECRET_NAME="AZURE-ML-SUBSCRIPTION-ID" 
AZURE_ML_RESOURCE_GROUP_SECRET_NAME="AZURE-ML-RESOURCE-GROUP" 
AZURE_ML_WORKSPACE_NAME_SECRET_NAME="AZURE-ML-WORKSPACE-NAME"

credential = DefaultAzureCredential()
ml_client = MLClient.from_config(credential=credential)
keyvault_name = ml_client.workspaces.get(ml_client.workspace_name).key_vault.split('/')[-1]
kv_url = f"https://{keyvault_name}.vault.azure.net/"



def readSecret(
    key_vault_url: str,
    secret_name: str
) -> str:
    """
    read secret in keyvault .

    Args:
        key_vault_url:  Key Vault url.
        secret_name:    Name of the secret
    Returns:
        secret value (str).
    """        
    kv_client = SecretClient(vault_url=key_vault_url, credential=credential)
    secret_value = kv_client.get_secret(secret_name).value
    return secret_value

global_model_id = readSecret(kv_url,AZURE_ML_MODEL_EMPTY_ID_SECRET_NAME)
global_model_id=os.getenv("AZURE_ML_MODEL_ID",default=global_model_id)
print(f"global_model_id:{global_model_id}")

local_model_name = readSecret(kv_url,AZURE_ML_MODEL_EMPTY_NAME_SECRET_NAME)
local_model_name=os.getenv("AZURE_ML_MODEL_NAME",default=local_model_name)
print(f"local_model_name:{local_model_name}")

global_instance_type = readSecret(kv_url,AZURE_ML_MODEL_EMPTY_INSTANCE_TYPE_SECRET_NAME)
global_instance_type=os.getenv("AZURE_ML_INSTANCE_TYPE",default=global_instance_type)
print(f"global_instance_type:{global_instance_type}")

global_score_script = readSecret(kv_url,AZURE_ML_MODEL_EMPTY_SCORE_SCRIPT_SECRET_NAME)
global_score_script=os.getenv("AZURE_ML_SCORING_SCRIPT",default=global_score_script)
print(f"global_score_script:{global_score_script}")


prefix = readSecret(kv_url,AZURE_ML_MODEL_PREFIX_SECRET_NAME)
global_model_name = f"{prefix}-{local_model_name}"
global_model_name=os.getenv("AZURE_ML_MODEL_NAME",default=global_model_name)
print(f"global_model_name:{global_model_name}")

prefix = readSecret(kv_url,AZURE_ML_ENDPOINT_PREFIX_SECRET_NAME)
global_endpoint_name = f"{prefix}-{local_model_name}"
global_endpoint_name=os.getenv("AZURE_ML_ENDPOINT_NAME",default=global_endpoint_name)
print(f"global_endpoint_name:{global_endpoint_name}")

prefix = readSecret(kv_url,AZURE_ML_ENV_PREFIX_SECRET_NAME)
global_env_name = f"{prefix}-{local_model_name}"
global_env_name=os.getenv("AZURE_ML_ENV_NAME",default=global_env_name)
print(f"global_env_name:{global_env_name}")


global_subscription_id = readSecret(kv_url,AZURE_ML_SUBSCRIPTION_ID_SECRET_NAME)
global_subscription_id=os.getenv("AZURE_ML_SUBSCRIPTION_ID",default=global_subscription_id)
print(f"global_subscription_id:{global_subscription_id}")
    
global_resource_group_name = readSecret(kv_url,AZURE_ML_RESOURCE_GROUP_SECRET_NAME)
global_resource_group_name=os.getenv("AZURE_ML_RESOURCE_GROUP",default=global_resource_group_name)
print(f"global_resource_group_name:{global_resource_group_name}")

global_workspace_name = readSecret(kv_url,AZURE_ML_WORKSPACE_NAME_SECRET_NAME)
global_workspace_name=os.getenv("AZURE_ML_WORKSPACE_NAME",default=global_workspace_name)
print(f"global_workspace_name:{global_workspace_name}")

## Download and Save the Qwen2-1.5B Model

Downloads the Qwen2-1.5B model weights and tokenizer from the Hugging Face Hub using the model ID resolved from Key Vault, then saves both to the local `./model` directory. The saved artefacts are used in subsequent cells to register the model in Azure ML and as the serving artefact inside the online deployment.

In [ ]:
from transformers import AutoModel, AutoTokenizer

model_id = global_model_id
model = AutoModel.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

model.save_pretrained("./model")
tokenizer.save_pretrained("./model")

## Register the Model in Azure ML

Creates a new Azure ML `MLClient` scoped to the target subscription, resource group, and workspace, then registers the locally saved `./model` directory as a `custom_model` asset under the name derived from Key Vault configuration. Registering the model creates a versioned, trackable artefact in the Azure ML model registry that can be referenced by deployments.

In [ ]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Model
from azure.identity import DefaultAzureCredential

subscription_id = global_subscription_id
resource_group = global_resource_group_name
workspace_name = global_workspace_name

ml_client = MLClient(
    DefaultAzureCredential(),
    subscription_id,
    resource_group,
    workspace_name,
)

model = Model(
    name=global_model_name,
    path="./model",
    type="custom_model",
)

model = ml_client.models.create_or_update(model)


## Create a Managed Online Endpoint

Provisions a new Azure ML `ManagedOnlineEndpoint` with key-based authentication. The endpoint name is made unique by appending a timestamp to the configured prefix, preventing naming collisions on repeated runs. The call blocks until the endpoint is fully provisioned before the notebook continues.

In [ ]:
from azure.ai.ml.entities import ManagedOnlineEndpoint
import datetime

endpoint_name = global_endpoint_name

endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    auth_mode="key",  # or "aad_token"
    description=f"{global_model_name} real-time inference endpoint",
)

ml_client.online_endpoints.begin_create_or_update(endpoint).result()

## Define and Test the Scoring Functions Locally

Defines the `init()` and `run()` functions that constitute the scoring script served by Azure ML:
- **`init()`** loads the `OPTForCausalLM` model and tokenizer from the local `./model` directory into memory.
- **`run(raw_data)`** accepts a JSON string with a `"text"` key, generates a continuation using the model (up to 50 new tokens, temperature 0.7), and returns the decoded result.

The cell then calls both functions with a sample prompt to validate the scoring logic before it is packaged and deployed to the online endpoint.

In [ ]:
import os
import logging

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

def init():

    # base = os.environ["AZUREML_MODEL_DIR"]              # e.g. /var/azureml-app/azureml-models/.../1
    # model_dir = os.path.join(base, "model")             # <- because your files are in .../1/model/
    model_dir = "./model" 
    # logger.info(f"AZUREML_MODEL_DIR={base}")
    logger.info(f"Using model_dir={model_dir}")
    logger.info(f"Files in model_dir: {os.listdir(model_dir)}")


def run(raw_data):
    logger.info(f"input={raw_data}")
    result = "this is my response"
    logger.info(f"generated_text={result}")
    return {   "choices": [
            {
            "index": 0,
            "message": {
                "role": "assistant",
                "content": result
            },
            "finish_reason": "stop"
            }
            ]
        }
init()
input_data = "{\"text\": \"The future of artificial intelligence is\"}"
print(f"input_data: {input_data}")
result = run(input_data)
print(f"result: {result}")

## Register the Conda Environment in Azure ML

Builds an Azure ML `Environment` that combines a minimal Ubuntu 22.04 CPU inference base image with a Conda specification that installs PyTorch, Transformers, and the Azure ML inference HTTP server. The environment is registered (or a new version created if it already exists) in the workspace, and all available versions are listed for reference.

In [ ]:
import yaml
from azure.ai.ml.entities import Environment

conda_spec = yaml.safe_load(f"""
name: {global_env_name}
channels:
- conda-forge
dependencies:
- python=3.10
- pip
- pip:
  - torch
  - transformers
  - azureml-inference-server-http
""")

env = Environment(
    name=global_env_name,
    description="Torch + Transformers on minimal inference image",
    image="mcr.microsoft.com/azureml/minimal-ubuntu22.04-py39-cpu-inference:latest",
    conda_file=conda_spec,
)

registered_env = ml_client.environments.create_or_update(env)
print("Registered:", registered_env.name, registered_env.version)

# List all versions
for e in ml_client.environments.list(name=global_env_name):
  print(e.name, e.version)

# Or get the one you just created
# ml_client.environments.get(name=registered_env.name, version=registered_env.version)


## Create the Online Deployment

Creates a `ManagedOnlineDeployment` named `"blue"` that attaches the registered model, the registered Conda environment, and the scoring script from the `./src` directory to the previously created endpoint. Liveness and readiness probes are configured with a 300-second initial delay to allow sufficient time for the large model to load before health checks begin. The call blocks until the deployment is fully active.

In [ ]:
from azure.ai.ml.entities import ManagedOnlineDeployment, CodeConfiguration, ProbeSettings
print(f"endpoint: {endpoint_name}")
print(f"env:{registered_env.name}:{registered_env.version}")
deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=endpoint_name,
    model=model,                 # or "hf-model:1"
    environment=f"{registered_env.name}:{registered_env.version}",  # or "hf-env@latest"
    code_configuration=CodeConfiguration(code="./src", scoring_script=global_score_script),
    instance_type=global_instance_type,  # e.g. "Standard_DS3_v2"
    instance_count=1,    
    environment_variables={
        "AZUREML_HTTP_PORT": "5001"
    },
    liveness_probe=ProbeSettings(
        initial_delay=300,
        timeout=10,
        period=10,
        failure_threshold=6,
    ),
    readiness_probe=ProbeSettings(
        initial_delay=300,
        timeout=10,
        period=10,
        failure_threshold=6,
    ),


)

ml_client.online_deployments.begin_create_or_update(deployment).result()

## Route 100% of Traffic to the Blue Deployment

Fetches the current endpoint configuration and updates its traffic map so that all incoming requests are directed to the `"blue"` deployment. This step is required after deployment creation because Azure ML endpoints default to 0% traffic on newly created deployments.

In [ ]:
# Route traffic to the deployment
endpoint = ml_client.online_endpoints.get(name=endpoint_name)
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

## Print Azure CLI Commands to Test the Endpoint

Prints ready-to-run Azure CLI commands that can be copy-pasted into a terminal to:
1. Retrieve the endpoint's scoring URI.
2. Retrieve the primary API key.
3. Send a test inference request using `curl`.
4. Fetch the last 200 lines of deployment logs for troubleshooting.

All commands are pre-filled with the endpoint name, resource group, and workspace name resolved earlier in the notebook.

In [ ]:
print("# Run the following commands to test the endpoint with the Azure CLI:")
print(f"URI=\$(az ml online-endpoint show --name {endpoint_name}   --resource-group {global_resource_group_name}   --workspace-name {global_workspace_name}  --query scoring_uri -o tsv)")
print(f"TOKEN=\$(az ml online-endpoint get-credentials   --name {endpoint_name}   --resource-group {global_resource_group_name}   --workspace-name {global_workspace_name} --query primaryKey -o tsv)")
print(f"curl -X POST \"\$URI\"     -H \"Content-Type: application/json\"     -H \"Authorization: Bearer \$TOKEN\"     -d '{{\"text\": \"I am going to Paris, what should I see?\"}}'")
print("# Run the following commands to get the logs with the Azure CLI:")
print(f"az ml online-deployment get-logs   --name 'blue' --lines 200 --endpoint-name {endpoint_name}   --resource-group {global_resource_group_name}   --workspace-name {global_workspace_name}")